In [8]:
import pandas as pd
import numpy as np
from datetime import datetime
import re
import sys
import matplotlib.pyplot as plt
from functools import reduce
import glob, os

In [9]:
RUTA = "Valuaciones_USA"
# Excluye archivos temporales de Excel (prefijo ~$)
files = sorted([f for f in glob.glob(f"{RUTA}/*_score_ponderados.xlsx") if not os.path.basename(f).startswith("~")])

registros = []
for f in files:
    ticker = os.path.basename(f).split("_score_ponderados")[0]
    df = pd.read_excel(f, index_col=0)

    # Columnas con formato MesAño (ej. Dec2024, Aug2025)
    cols_periodo = [c for c in df.columns if isinstance(c, str) and re.match(r"^[A-Za-z]{3}\d{4}$", c)]

    # Últimos 3 disponibles
    ultimos_3 = cols_periodo[-3:]

    # Fila Score Total
    score_row = df[df.index.astype(str).str.contains("Score Total", case=False, na=False)]

    for col in ultimos_3:
        valor = score_row[col].values[0] if not score_row.empty else None
        mes = col[:3]
        anio = int(col[3:])
        registros.append({"Ticker": ticker, "Periodo": col, "Mes": mes, "Año": anio, "Score_Total": valor})

df_scores = pd.DataFrame(registros)
df_scores["Score_Total"] = pd.to_numeric(df_scores["Score_Total"], errors="coerce").round(2)


In [ ]:
# Último año disponible por empresa
ultimo_anio = df_scores.groupby("Ticker")["Año"].max().reset_index()
ultimo_score = df_scores.merge(ultimo_anio, on=["Ticker", "Año"])

# Empresas con score > 60 en su último año
tickers_validos = ultimo_score[ultimo_score["Score_Total"] > 60]["Ticker"].tolist()

df_scores_filtrado = df_scores[df_scores["Ticker"].isin(tickers_validos)].reset_index(drop=True)

print(f"Empresas que pasan el filtro: {len(tickers_validos)}")
print(sorted(tickers_validos))
df_scores_filtrado

Empresas que pasan el filtro: 17
['ACN', 'ADBE', 'AMZN', 'CRM', 'GOOG', 'INTU', 'MA', 'MELI', 'META', 'MNST', 'MSFT', 'NFLX', 'NKE', 'PCAR', 'TER', 'TSM', 'V']


,Ticker,Periodo,Mes,Año,Score_Total
0,ACN,Aug2023,Aug,2023,77.00
1,ACN,Aug2024,Aug,2024,70.50
2,ACN,Aug2025,Aug,2025,70.25
3,ADBE,Nov2021,Nov,2021,81.50
4,ADBE,Nov2022,Nov,2022,83.75
5,ADBE,Nov2023,Nov,2023,82.50
6,AMZN,Dec2022,Dec,2022,25.50
7,AMZN,Dec2023,Dec,2023,42.25
8,AMZN,Dec2024,Dec,2024,62.00
9,CRM,Jan2023,Jan,2023,45.00
